In [4]:
import glob
import re
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import yt

yt.set_log_level(50)

# 1. Gather and sort plot directories
plot_dirs = sorted(glob.glob("../raw/test*"))


def extract_step(path):
    match = re.search(r"test(\d+)", path)
    return int(match.group(1)) if match else -1


plot_dirs.sort(key=extract_step)

if not plot_dirs:
    raise FileNotFoundError("No directories matching '../raw/test*' were found.")

# Load datasets
ds_list = [yt.load(d) for d in plot_dirs]
times = [float(ds.current_time) for ds in ds_list]

# Get z-bounds from the first dataset
z_min = float(ds_list[0].domain_left_edge[2])
z_max = float(ds_list[0].domain_right_edge[2])

# Map field tuples to dropdown options
raw_fields = ds_list[0].field_list
field_options = [(f[1] if isinstance(f, tuple) else str(f), f) for f in raw_fields]

# 2. Build widgets
step_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(plot_dirs) - 1,
    step=1,
    description="Step Index:",
    continuous_update=False,
)

z_slider = widgets.FloatSlider(
    value=(z_min + z_max) / 2.0,
    min=z_min,
    max=z_max,
    step=(z_max - z_min) / 100.0,
    description="z-Slice (m):",
    continuous_update=False,
)

field_dropdown = widgets.Dropdown(
    options=field_options,
    description="Field:",
)


# 3. Render function
def plot_slice(step_idx, z_val, field):
    ds = ds_list[step_idx]
    current_t = times[step_idx]

    p = yt.SlicePlot(ds, "z", field, center=[0.0, 0.0, z_val])

    # Force linear scale so yt doesn't attempt symlog linthresh estimation on zero/flat data
    p.set_log(field, False)

    # Compute symmetric limits around 0.0 for the current slice
    slc = ds.slice("z", z_val)
    data = slc[field].v
    finite_data = data[np.isfinite(data)]

    if len(finite_data) > 0 and not np.all(finite_data == 0):
        max_abs = float(np.max(np.abs(finite_data)))
    else:
        max_abs = 1.0  # Fallback range for uniform zero slice

    # Apply colormap, symmetric limits around 0, and set size
    p.set_cmap(field, "berlin")
    p.set_zlim(field, -max_abs, max_abs)
    p.set_figure_size(5.0)

    p.annotate_title(f"t = {current_t:.4f} | z = {z_val:.4f}")

    # Build Matplotlib plots without using yt's dynamic HTML backend
    p._setup_plots()

    # Retrieve and display the underlying Matplotlib figure directly
    fig = p.plots[field].figure
    display(fig)
    plt.close(fig)


# 4. Interactive Display
interactive_plot = widgets.interactive(
    plot_slice, step_idx=step_slider, z_val=z_slider, field=field_dropdown
)

display(interactive_plot)

interactive(children=(IntSlider(value=0, continuous_update=False, description='Step Index:'), FloatSlider(valu…

In [5]:
import glob
import os
import re
import imageio.v2 as imageio
import numpy as np
import yt
import shutil

yt.set_log_level(50)

# Setup directories
input_pattern = "../raw/test*"
temp_dir = "./temp_pngs"
output_gif_dir = "../gif"
output_gif_path = os.path.join(output_gif_dir, "nl_grad_damped_wave_equation.gif")

# Clean out existing temp folder if present, then recreate it empty
if os.path.exists(temp_dir):
    shutil.rmtree(temp_dir)
os.makedirs(temp_dir, exist_ok=True)
os.makedirs(output_gif_dir, exist_ok=True)


# 1. Gather and sort plot directories
def extract_step(path):
    match = re.search(r"test(\d+)", path)
    return int(match.group(1)) if match else -1


plot_dirs = sorted(glob.glob(input_pattern), key=extract_step)

if not plot_dirs:
    raise FileNotFoundError(f"No directories matching '{input_pattern}' were found.")

print(f"Found {len(plot_dirs)} plot directories. Processing slices at z=0...")

# Select field (e.g., first available field from the first dataset)
sample_ds = yt.load(plot_dirs[0])
target_field = sample_ds.field_list[1]
print(f"Plotting field: {target_field}")

png_files = []

# 2. Render each step to PNG
for idx, d in enumerate(plot_dirs):
    ds = yt.load(d)
    current_t = float(ds.current_time)
    step_num = extract_step(d)

    p = yt.SlicePlot(ds, "z", target_field, center=[0.0, 0.0, 0.0])
    p.set_log(target_field, False)

    # Compute symmetric limits around 0.0
    slc = ds.slice("z", 0.0)
    data = slc[target_field].v
    finite_data = data[np.isfinite(data)]

    if len(finite_data) > 0 and not np.all(finite_data == 0):
        max_abs = float(np.max(np.abs(finite_data)))
    else:
        max_abs = 1.0
    max_abs = 1.0

    p.set_cmap(target_field, "berlin")
    p.set_zlim(target_field, -max_abs, max_abs)
    p.set_figure_size(5.0)
    p.annotate_title(f"t = {current_t:.4f} | z = 0.0")

    # Save temporary frame
    frame_filename = os.path.join(temp_dir, f"frame_{step_num:05d}.png")
    p.save(frame_filename)
    png_files.append(frame_filename)

    print(f"[{idx+1}/{len(plot_dirs)}] Saved {frame_filename}", end="\r")

print("\nAll frames saved. Assembling GIF...")

# 3. Create GIF using imageio
# fps=10 gives a smooth animation; adjust fps as needed
images = [imageio.imread(f) for f in png_files]
imageio.mimsave(output_gif_path, images, fps=20, loop=0)

print(f"Successfully generated GIF: {output_gif_path}")

Found 401 plot directories. Processing slices at z=0...
Plotting field: ('boxlib', 'phi')
[401/401] Saved ./temp_pngs/frame_00800.png
All frames saved. Assembling GIF...
Successfully generated GIF: ../gif/nl_grad_damped_wave_equation.gif
